In [ ]:
folder_path = '../Datasets/Per_Attack_Datasets/'
dataframes = read_data(folder_path)

features = ['epre','pusch_snr','p_ue','ul_mcs','cqi','ul_bitrate',
            'dl_mcs','dl_retx','ul_tx','dl_tx','ul_retx','dl_bitrate','dl_err','ul_err'] # features in the dataset
num_features = 14 # number of features 
ws = 3

In [ ]:
import pandas as pd
import numpy as np
import csv

# Function to create sequences for a single user-event group
def create_sequences_for_group(group, features, target, sequence_length, writer):
    """
    For each sliding window in 'group', write one flattened sequence + label row to CSV.
    """
    for i in range(len(group) - sequence_length + 1):
        seq_features = group[features].iloc[i:i + sequence_length].values
        seq_label = group[target].iloc[i + sequence_length - 1]
        seq_mclabel = group[mc_label].iloc[i + sequence_length - 1]
        
        # flatten sequence and append label
        flat = seq_features.reshape(-1)
        row = np.concatenate([flat, [seq_label, seq_mclabel]])
        writer.writerow(row)

In [ ]:
attacks = ['SYN', 'ICMP', 'UDP', 'DNS', 'GTPU'] 
for attack in attacks:
    df = pd.read_csv(folder_path+attack+".csv")
    out_path = "../Datasets/sequences_mc/"+attack+"_mc.csv"
    data = df.drop(columns=['attack', 'malicious', 'attack_number'], errors='ignore')

    # Parse and sort data
    data['_time'] = pd.to_datetime(data['_time'])
    data.sort_values(by=['imeisv', '_time'], inplace=True)
    
    target = "binary_label"
    mc_label = "multiclass_label"
    
    # Build column names: feature_t0, feature_t1, ..., feature_t(ws-1), label
    cols = [f"{feat}_t{t}" for t in range(ws) for feat in features] + ["label"] + ["multiclass_label"]

    with open(out_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(cols)  # header row
    
        # Loop over groups and write sequences directly inside the function
        for (imeisv, event), group in data.groupby(["imeisv", "event"]):
            if len(group) >= ws:
                create_sequences_for_group(group, features, target, ws, writer)
    
    print(f"✅ Sequences written directly to {out_path}")